In [17]:
import duckdb
import pandas as pd

In [18]:
##Create conection
conn = duckdb.connect("../ecommerce.duckdb")


In [19]:
##Testing conection
conn.execute("SELECT 1").fetchall()

[(1,)]

## Tables Creation

In [20]:
conn.execute("""
CREATE TABLE dim_customers (
    customer_id INTEGER,
    first_name VARCHAR,
    last_name VARCHAR,
    email VARCHAR,
    country VARCHAR,
    city VARCHAR,
    signup_date DATE,
    customer_segment VARCHAR
);

CREATE TABLE dim_categories (
    category_id INTEGER,
    category_name VARCHAR
);

CREATE TABLE dim_brands (
    brand_id INTEGER,
    brand_name VARCHAR
);

CREATE TABLE dim_products (
    product_id INTEGER,
    product_name VARCHAR,
    category_id INTEGER,
    brand_id INTEGER,
    price DECIMAL(10,2),
    cost DECIMAL(10,2),
    created_at TIMESTAMP
);

CREATE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    order_date TIMESTAMP,
    order_status VARCHAR,
    payment_method VARCHAR,
    shipping_country VARCHAR
);

CREATE TABLE fact_order_items (
    order_item_id INTEGER,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price DECIMAL(10,2),
    discount DECIMAL(5,2)
);
""")

CatalogException: Catalog Error: Table with name "dim_customers" already exists!

## Verify Tables

In [ ]:
conn.execute("SHOW TABLES").fetchdf()

,name
0,dim_brands
1,dim_categories
2,dim_customers
3,dim_products
4,fact_order_items
5,orders


### Upload all the CSV files to DuckDB
###### It creates 6 tables

In [ ]:
conn.execute("""
COPY dim_customers
FROM '../data/raw/dim_customers.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_categories
FROM '../data/raw/dim_categories.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_brands
FROM '../data/raw/dim_brands.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY dim_products
FROM '../data/raw/dim_products.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY orders
FROM '../data/raw/orders.csv'
(HEADER, DELIMITER ',');
""")

conn.execute("""
COPY fact_order_items
FROM '../data/raw/fact_order_items.csv'
(HEADER, DELIMITER ',');
""")

### Verify the rows of the fact_order_items table

In [ ]:
conn.execute("""
SELECT COUNT(*)
FROM fact_order_items
""").fetchdf()

,count_star()
0,60490


### First Join

In [ ]:
conn.execute("""
SELECT
    p.product_name,
    SUM(f.quantity * f.unit_price) AS revenue
FROM fact_order_items f
JOIN dim_products p
    ON f.product_id = p.product_id
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 10
""").fetchdf()

,product_name,revenue
0,Wind,1292415.56
1,There,1091420.64
2,Arrive,1075528.76
3,Become,1021696.32
4,Seek,972183.20
5,Contain,970397.24
6,Up,955946.80
7,Certainly,914287.72
8,Trouble,882641.20
9,Could,875513.84
